# Ground-radar ↔ ground-radar volume matching

A prototype that mirrors the `gpmmatch` satellite↔GR technique, but matches **two ground radars** to each other inside their overlapping coverage. The goal is a relative reflectivity calibration offset (in dB) between Radar A and Radar B.

## How it differs from the satellite case

In `gpmmatch`, the satellite nadir beam provides a privileged geometry and the match is done tilt-by-tilt in 2D (`gpmmatch.py:376`). Two ground radars are both *cones*, so:

- **No privileged geometry** → we use one common azimuthal-equidistant projection centered on the **midpoint** of the two sites, and match in full **3D**.
- **No parallax** → `correct.correct_parallax` is dropped.
- **Resolution volumes grow with range for *both* radars** → a fair comparison only exists in the near-equidistant zone, enforced by a **volume-ratio filter** (the main scientific addition here).
- **Different bands (S/C/X)** → reflectivity differs at high Z. We rely on the `correct.get_offset` masking of `> 36 dBZ` (near-Rayleigh regime) for a first pass, and leave a clearly-marked DFR hook for a cross-band correction later.
- **Asynchronous scans** → pick the closest-in-time volume scans; optional advection correction via `correct.grid_displacement`.

## Reused from `gpmmatch`
`compute_gaussian_curvature` (beam-height geometry), `get_offset` (calibration offset estimator), `attenuation_correction_*`, `grid_displacement` (advection), and the KDTree + Gaussian/volume weighting pattern from the main loop.

In [ ]:
import numpy as np
import pyodim
import pyproj
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

from gpmmatch import correct  # reuse compute_gaussian_curvature, get_offset, etc.

%matplotlib inline

## Configuration

Edit these for your two ODIM HDF5 files. Pick the two volume scans closest in time.

In [ ]:
# --- Inputs -------------------------------------------------------------
FILE_A = "radar_A.h5"        # ODIM HDF5, e.g. S-band
FILE_B = "radar_B.h5"        # ODIM HDF5, e.g. C-band

REFL_NAME = "DBZH"           # reflectivity field name in both files
BAND_A, BAND_B = "S", "C"    # frequency bands (for documentation / optional DFR / attenuation)
BEAMWIDTH_A = 1.0           # 3 dB beamwidth, degrees
BEAMWIDTH_B = 1.0

# --- Matching parameters ------------------------------------------------
RMIN = 15e3                 # skip cone-of-silence / near-range clutter (m)
RMAX_A = 150e3             # usable max range of radar A (m)
RMAX_B = 150e3             # usable max range of radar B (m)
REFL_THLD = 10.0           # minimum reflectivity to keep (dBZ)
VOL_RATIO_MAX = 2.0        # keep matches where 1/x < V_A/V_B < x (comparable volumes)
DZ_MAX = 500.0             # max height difference between matched volumes (m)
MIN_NEIGHBOURS = 5         # min radar-B gates inside a matched volume
A_STRIDE = 1               # subsample A gates for speed (use >1 on dense data)

## Step 1 — Read both radars

`pyodim.read_odim` returns one dataset per tilt with `x`, `y`, `range`, `azimuth`, `elevation`, the reflectivity field, and site attributes (`longitude`, `latitude`, `height`). We drop near-vertical birdbath tilts (`elevation > 80`), as `io.read_radar` does.

In [ ]:
def read_gr(grfile):
    nradar = pyodim.read_odim(grfile, lazy_load=False)
    if nradar[-1].elevation.max() > 80:
        nradar.pop(-1)
    return nradar

nradar_a = read_gr(FILE_A)
nradar_b = read_gr(FILE_B)

lon_a, lat_a = nradar_a[0].attrs["longitude"], nradar_a[0].attrs["latitude"]
lon_b, lat_b = nradar_b[0].attrs["longitude"], nradar_b[0].attrs["latitude"]

# Baseline distance between the two sites
geod = pyproj.Geod(ellps="WGS84")
_, _, baseline = geod.inv(lon_a, lat_a, lon_b, lat_b)
print(f"Radar A: ({lat_a:.4f}, {lon_a:.4f})  {len(nradar_a)} tilts, band {BAND_A}")
print(f"Radar B: ({lat_b:.4f}, {lon_b:.4f})  {len(nradar_b)} tilts, band {BAND_B}")
print(f"Baseline distance: {baseline / 1e3:.1f} km")

## Step 2 — Common projection and gate extraction

One aeqd projection centered on the **midpoint** so neither radar is privileged. For each gate we compute:

- common `(x, y)` — from the radar-local `x,y` offset by the site position in the common projection (small-angle approximation, fine over ~100 km baselines);
- height `z` — from slant range + tilt elevation under the 4/3-earth model (reusing `compute_gaussian_curvature`);
- resolution volume `vol = dr · (r·θ)²` — same definition as `gpmmatch.py:320`.

> **Cross-band hook:** apply attenuation correction here for C/X (e.g. `correct.attenuation_correction_zphi`) and/or a DFR conversion if you later want to correct Z to a common band. Left out of this first pass.

In [ ]:
# Common projection centered on the midpoint of the two sites
lon0, lat0 = 0.5 * (lon_a + lon_b), 0.5 * (lat_a + lat_b)
proj = pyproj.Proj(f"+proj=aeqd +lon_0={lon0} +lat_0={lat0} +ellps=WGS84")
ae = correct.compute_gaussian_curvature(lat0)  # 4/3-earth effective radius (m)

site_a = np.array(proj(lon_a, lat_a))
site_b = np.array(proj(lon_b, lat_b))


def extract_gates(nradar, refl_name, site_xy, beamwidth):
    """Flatten all tilts to 1D arrays in the common projection."""
    alt = nradar[0].attrs.get("height", 0.0)
    bw = np.deg2rad(beamwidth)
    X, Y, Z, ZH, VOL, RNG = [], [], [], [], [], []
    for tilt in nradar:
        rng = tilt.range.values
        dr = float(rng[1] - rng[0])
        elev = float(tilt.elevation.values[0])
        xl, yl = tilt.x.values, tilt.y.values  # radar-local (site at 0,0)
        # Gate height (4/3-earth model); depends only on range and elevation
        z1d = np.sqrt(rng**2 + ae**2 + 2 * rng * ae * np.sin(np.deg2rad(elev))) - ae + alt
        z2d = np.broadcast_to(z1d[np.newaxis, :], xl.shape)
        r2d = np.broadcast_to(rng[np.newaxis, :], xl.shape)
        vol = 1e-9 * dr * (r2d * bw) ** 2
        zh = tilt[refl_name].values
        X.append((site_xy[0] + xl).ravel())
        Y.append((site_xy[1] + yl).ravel())
        Z.append(z2d.ravel())
        ZH.append(np.asarray(zh, dtype=float).ravel())
        VOL.append(vol.ravel())
        RNG.append(r2d.ravel())
    return {k: np.concatenate(v) for k, v in
            dict(x=X, y=Y, z=Z, zh=ZH, vol=VOL, rng=RNG).items()}


A = extract_gates(nradar_a, REFL_NAME, site_a, BEAMWIDTH_A)
B = extract_gates(nradar_b, REFL_NAME, site_b, BEAMWIDTH_B)
print(f"Radar A gates: {A['x'].size:,}   Radar B gates: {B['x'].size:,}")

## Step 3 — Overlap region

Keep gates within `[RMIN, RMAX]` of **both** sites. Visualise the footprint as a sanity check before matching.

In [ ]:
def dist_to(site, d):
    return np.hypot(d["x"] - site[0], d["y"] - site[1])

# A gate is usable if it is in range of its own radar AND of the other radar
in_overlap_a = ((dist_to(site_a, A) > RMIN) & (dist_to(site_a, A) < RMAX_A) &
                (dist_to(site_b, A) > RMIN) & (dist_to(site_b, A) < RMAX_B) &
                np.isfinite(A["zh"]) & (A["zh"] > REFL_THLD))
in_overlap_b = ((dist_to(site_a, B) > RMIN) & (dist_to(site_a, B) < RMAX_A) &
                (dist_to(site_b, B) > RMIN) & (dist_to(site_b, B) < RMAX_B) &
                np.isfinite(B["zh"]) & (B["zh"] > REFL_THLD))
print(f"A gates in overlap with echo: {in_overlap_a.sum():,}")
print(f"B gates in overlap with echo: {in_overlap_b.sum():,}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(A["x"][in_overlap_a] / 1e3, A["y"][in_overlap_a] / 1e3, s=1, alpha=0.2, label="A echo")
ax.scatter(B["x"][in_overlap_b] / 1e3, B["y"][in_overlap_b] / 1e3, s=1, alpha=0.2, label="B echo")
ax.plot(*(site_a / 1e3), "k^", ms=12, label="Radar A")
ax.plot(*(site_b / 1e3), "r^", ms=12, label="Radar B")
ax.set_xlabel("x (km)"); ax.set_ylabel("y (km)"); ax.set_aspect("equal")
ax.legend(); ax.set_title("Overlap region (echo gates)")
plt.show()

## Step 4 — 3D volume matching

For each Radar-A gate in the overlap, find Radar-B gates inside a matched volume (3D KDTree query), then beam-weight B's reflectivity by gate volume and Gaussian distance — the same weighting as `gpmmatch.py:425`. The search radius scales with the larger of the two beam diameters at that location, so it adapts to range.

We keep a match only where the two resolution volumes are comparable (`VOL_RATIO_MAX`) and the heights agree (`DZ_MAX`).

In [ ]:
# Restrict to overlap gates and build a 3D tree on radar B
idx_a = np.where(in_overlap_a)[0][::A_STRIDE]
Bx, By, Bz = B["x"][in_overlap_b], B["y"][in_overlap_b], B["z"][in_overlap_b]
Bzh, Bvol = B["zh"][in_overlap_b], B["vol"][in_overlap_b]
treeB = cKDTree(np.column_stack([Bx, By, Bz]))

bw_b = np.deg2rad(BEAMWIDTH_B)
bw_a = np.deg2rad(BEAMWIDTH_A)

rows = []  # (z_a, z_b_weighted, height, vol_a, vol_b, n)
for i in idx_a:
    pa = np.array([A["x"][i], A["y"][i], A["z"][i]])
    rng_b_here = np.hypot(pa[0] - site_b[0], pa[1] - site_b[1])
    # search radius ~ half the larger beam diameter at this location
    search = 0.5 * max(bw_a * A["rng"][i], bw_b * rng_b_here)
    nb = treeB.query_ball_point(pa, search)
    if len(nb) < MIN_NEIGHBOURS:
        continue
    zb = Bzh[nb]
    valid = np.isfinite(zb) & (zb > REFL_THLD)
    if valid.sum() < MIN_NEIGHBOURS:
        continue
    nb = np.asarray(nb)[valid]
    d = np.sqrt((Bx[nb] - pa[0])**2 + (By[nb] - pa[1])**2 + (Bz[nb] - pa[2])**2)
    w = Bvol[nb] * np.exp(-(d / search) ** 2)
    z_b = np.sum(w * Bzh[nb]) / np.sum(w)  # beam-weighted mean (dBZ), matches gpmmatch
    rows.append((A["zh"][i], z_b, A["z"][i], A["vol"][i], Bvol[nb].mean(), len(nb)))

rows = np.array(rows)
print(f"Raw matched volumes: {len(rows):,}")

## Step 5 — Fair-comparison filter, scatter, and calibration offset

Apply the volume-ratio filter, then estimate the A−B offset. `correct.get_offset` expects a matchset with specific variable names, so here we inline its histogram-overlap logic directly on the two matched arrays (same algorithm, `correct.py:369-377`).

In [ ]:
z_a, z_b, height, vol_a, vol_b, n = rows.T
ratio = vol_a / vol_b
keep = (ratio > 1 / VOL_RATIO_MAX) & (ratio < VOL_RATIO_MAX)
za, zb = z_a[keep], z_b[keep]
print(f"Matched volumes after volume-ratio filter: {keep.sum():,}")


def histogram_overlap_offset(ref, other, lo=0, hi=50, nbins=200, cap=36.0):
    """Mode of the histogram-overlap, as in correct.get_offset.
    Masks samples > cap to stay in the near-Rayleigh regime (key for cross-band)."""
    r, o = ref.copy(), other.copy()
    bad = (r > cap) | (o > cap)
    r[bad] = np.nan; o[bad] = np.nan
    offsets = np.arange(-15, 15, 0.2)
    pdf_ref, _ = np.histogram(r, range=(lo, hi), bins=nbins, density=True)
    area = np.array([np.sum(np.min([np.histogram(o - a, range=(lo, hi), bins=nbins, density=True)[0], pdf_ref], axis=0))
                     for a in offsets])
    smoothed = np.convolve([1] * 12, area, "same")
    return offsets[np.argmax(smoothed)]

offset = histogram_overlap_offset(za, zb)
print(f"Estimated B−A reflectivity offset: {offset:.2f} dB")
print(f"Mean(A)={np.nanmean(za):.2f}  Mean(B)={np.nanmean(zb):.2f}  "
      f"Median(B−A)={np.nanmedian(zb - za):.2f} dB")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.hist2d(za, zb, bins=60, range=[[0, 55], [0, 55]], cmin=1)
ax1.plot([0, 55], [0, 55], "k--")
ax1.set_xlabel(f"Radar A Z (dBZ, {BAND_A})"); ax1.set_ylabel(f"Radar B Z (dBZ, {BAND_B})")
ax1.set_title("Matched reflectivity"); ax1.set_aspect("equal")
ax2.hist(zb - za, bins=60, range=(-15, 15))
ax2.axvline(offset, color="r", label=f"offset={offset:.2f} dB")
ax2.set_xlabel("B − A (dB)"); ax2.set_ylabel("count"); ax2.legend()
plt.tight_layout(); plt.show()

## Notes & next steps

1. **Cross-band correction (S/C/X).** The offset above relies on masking `> 36 dBZ`. To use the full Z range, add a DFR conversion to a common band in Step 2 (adapt `correct.convert_gpmrefl_grband_dfr`, which is Ku→GR; a phase-aware version needs a melting-layer height — derivable from a sounding or each radar's own 0°C estimate).
2. **Attenuation.** For the C/X radar, apply `correct.attenuation_correction_zphi` (if KDP available) or `..._gunn_east` per tilt before flattening.
3. **Time / advection.** Confirm the two volume scans are close in time. For larger gaps, advect one field to the other's time with `correct.grid_displacement` (phase correlation on a common 2D grid).
4. **Symmetry check.** Re-run with A and B swapped; a clean calibration difference should be roughly antisymmetric.
5. **Performance.** The match loop is a Python `for` over A gates — fine for a single overlap, slow for archives. Increase `A_STRIDE`, or vectorise with `cKDTree.query_ball_point` on batched points once the logic is validated.
6. **Promotion to a module.** Once stable, fold this into `gpmmatch` as a `grmatch` sibling reusing `correct.py`, with an xarray output matching `volume_matching`'s schema.